In [ ]:
from pathlib import Path
from typing import Union

## Phase 1

### Defining the Motif Length (w)

In [ ]:
def read_protein_pattern(note_path: Path):
    parsed_file = []
    with open(note_path, "r") as note_file:
        data = note_file.read().strip().split("\n")
        for example in data:
            try:
                value = float(example.split("\t")[0])
            except Exception:
                value = None
            character = example.split("\t")[1]
            parsed_file.append((value, character))
    return parsed_file

protein_path_1 = Path("./data/AIBI-lab-01-data/RBP-footprinting-data/HNRNPA2B1/hnrnpa2b1_expected_pattern.txt")
motiff_1 = read_protein_pattern(protein_path_1)
motiff_1

In [ ]:
protein_path_2 = Path("./data/AIBI-lab-01-data/RBP-footprinting-data/HNRNPC/hnrnpc_expected_pattern.txt")
motiff_2 = read_protein_pattern(protein_path_2)
motiff_2

In [ ]:
def extract_motiff_length(motiff_representation: Union[list, tuple], expand: int=2):
    acceptable_lengths = []
    for i in range(expand+1):
        acceptable_lengths.append(len(motiff_representation)+i)
    return acceptable_lengths

motiff_1_lengths = extract_motiff_length(motiff_1)
motiff_1_lengths

In [ ]:
motiff_2_lengths = extract_motiff_length(motiff_2)
motiff_2_lengths

### Extracting Promising Motiffs

In [ ]:
import glob

def find_all_txts(dir: Path):
    txts = glob.glob("*.txt", root_dir=dir)
    return txts

binging_sites_dir = Path("./data/AIBI-lab-01-data/RBP-footprinting-data/HNRNPA2B1/hnrnpa2b1_binding_sites_fshape")
binding_sites_txts = find_all_txts(binging_sites_dir)
binding_sites_1 = []

for site in binding_sites_txts:
    full_path = Path(binging_sites_dir, site)
    out = read_protein_pattern(full_path)
    binding_sites_1.append(out)
binding_sites_1

In [ ]:
binging_sites_dir = Path("./data/AIBI-lab-01-data/RBP-footprinting-data/HNRNPC/hnrnpc_binding_sites_fshape")
binding_sites_txts = find_all_txts(binging_sites_dir)
binding_sites_2 = []

for site in binding_sites_txts:
    full_path = Path(binging_sites_dir, site)
    out = read_protein_pattern(full_path)
    binding_sites_2.append(out)
binding_sites_2

In [ ]:
def sliding_window_reactivity(protein_pattern: list, window_size: int, min_reactivity: float=1.0):
    results = []
    if window_size > len(protein_pattern):
        return results
    for i in range(len(protein_pattern)-window_size):
        protein_slice = protein_pattern[i: i+window_size]
        for val in protein_slice:
            if val[0] is not None and val[0]>= min_reactivity:
                results.append(protein_slice)
                break
    return results

promising_binding_sites_1 = {}
for window_size in motiff_1_lengths:
    promising_binding_sites_1[window_size] = []
    for binding_site in binding_sites_1:
        outs = sliding_window_reactivity(binding_site, window_size=window_size)
        if len(outs) == 0:
            continue
        for out in outs:
            promising_binding_sites_1[window_size].append(out)
promising_binding_sites_1

In [ ]:
promising_binding_sites_2 = {}
for window_size in motiff_2_lengths:
    promising_binding_sites_2[window_size] = []
    for binding_site in binding_sites_2:
        outs = sliding_window_reactivity(binding_site, window_size=window_size)
        if len(outs) == 0:
            continue
        for out in outs:
            promising_binding_sites_2[window_size].append(out)
promising_binding_sites_2